# Stock Market Data Collection

Historical stock market data is collected using the Yahoo Finance API through the yfinance library.

Dataset Includes:
- Open Price
- High Price
- Low Price
- Close Price
- Volume

The selected stock data is downloaded for a specified time period and stored in a pandas DataFrame for further processing.

In [5]:
import yfinance as yf
import pandas as pd

# Download Apple stock data
data = yf.download("AAPL", start="2020-01-01")

print(data.head())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.333885  72.394093  71.091191  71.344062  135480400
2020-01-03  71.630646  72.389265  71.406674  71.563213  146322800
2020-01-06  72.201424  72.239958  70.503561  70.754028  118387200
2020-01-07  71.861839  72.466322  71.642681  72.211041  108872000
2020-01-08  73.017822  73.318862  71.565606  71.565606  132079200


# Feature Engineering

Feature engineering is the process of creating meaningful indicators from raw stock market data.

Technical indicators help the model understand:
- Market trends
- Momentum
- Volatility
- Price movement patterns

Features Used:
- SMA (Simple Moving Average)
- EMA (Exponential Moving Average)
- RSI (Relative Strength Index)
- MACD
- Signal Line
- Volatility
- Momentum
- Volume Change

These features improve the model’s ability to learn hidden market behavior.

In [6]:
# Moving averages
data['SMA_20'] = data['Close'].rolling(20).mean()
data['EMA_20'] = data['Close'].ewm(span=20).mean()

# RSI
delta = data['Close'].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
data['RSI'] = 100 - (100 / (1 + rs))

# MACD
exp1 = data['Close'].ewm(span=12).mean()
exp2 = data['Close'].ewm(span=26).mean()

data['MACD'] = exp1 - exp2
data['Signal_Line'] = data['MACD'].ewm(span=9).mean()

# Returns
data['Return'] = data['Close'].pct_change()

# Volatility
data['Volatility'] = data['Return'].rolling(20).std()

# Momentum
data['Momentum'] = data['Close'] - data['Close'].shift(10)

# Volume Change
data['Volume_Change'] = data['Volume'].pct_change()

# Creating the Target Variable

The target variable represents the value the model will predict.

In this project:
- The target is the next-day stock return.

In [7]:
data['Target'] = data['Return'].shift(-1)

# Data Cleaning and Feature Selection

Before training the model, missing values generated during feature engineering are removed.

Data Cleaning:
- Removes NaN values
- Ensures clean and consistent input data

Feature Selection:
Only important technical indicators are selected as model inputs.

Selected features are stored in:
- X → Input features
- y → Target variable

This prepares the dataset for scaling and sequence generation.

In [10]:
data.dropna(inplace=True)

In [9]:
features = [
    'SMA_20',
    'EMA_20',
    'RSI',
    'MACD',
    'Signal_Line',
    'Volatility',
    'Momentum',
    'Volume_Change'
]

X = data[features]
y = data['Target']

In [11]:
import numpy as np

sequence_length = 10

X_seq, y_seq = [], []

for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

In [16]:
# !pip install torch

In [15]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = LSTMModel(input_size=X_seq.shape[2], hidden_size=50)

In [17]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    outputs = model(torch.tensor(X_seq).float())
    loss = criterion(outputs.squeeze(), torch.tensor(y_seq).float())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    